In [2]:
"""
Currently we have Query -> Hybrid Retrieval -> Top chunks
and then we will be having
Query -> Hybrid Retrieval -> Top 20 Chunks -> Cross Encoder Reranker -> Best 5 chunks

"""

'\nCurrently we have Query -> Hybrid Retrieval -> Top chunks \nand then we will be having \nQuery -> Hybrid Retrieval -> Top 20 Chunks -> Cross Encoder Reranker -> Best 5 chunks\n\n'

In [5]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-text-splitters \
faiss-cpu \
rank-bm25 \
sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
import pickle
import pandas as pd
import numpy as np

from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings

from sentence_transformers import CrossEncoder

from collections import defaultdict

/tmp/ipykernel_5417/1983733285.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading the previous saved data

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/legal_ai")
VECTOR_DIR = PROJECT_DIR / "vectorstores"
METADATA_DIR = PROJECT_DIR / "metadata"
FAISS_DIR = VECTOR_DIR / "legal_faiss"

In [14]:
print("Project:", PROJECT_DIR)
print("Vector Store:", VECTOR_DIR)
print("Metadata:", METADATA_DIR)
print("FAISS:", FAISS_DIR)

Project: /content/drive/MyDrive/legal_ai
Vector Store: /content/drive/MyDrive/legal_ai/vectorstores
Metadata: /content/drive/MyDrive/legal_ai/metadata
FAISS: /content/drive/MyDrive/legal_ai/vectorstores/legal_faiss


In [18]:
with open( "/content/drive/MyDrive/legal_ai/vectorstores/chunks.pkl","rb") as f:

    chunks = pickle.load(f)

print("Chunks Loaded:", len(chunks))

Chunks Loaded: 37818


In [19]:
import pandas as pd
metadata_df = pd.read_csv( "/content/drive/MyDrive/legal_ai/metadata/contract_type_mapping.csv")

print(metadata_df.shape)

metadata_df.head()

(510, 2)


,source,contract_type
0,"ADMA BioManufacturing, LLC - Amendment #3 to ...",Manufacturing Agreement
1,ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX...,Agency Agreement
2,"BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER ...",Supply Agreement
3,AlliedEsportsEntertainmentInc_20190815_8-K_EX-...,License Agreement
4,ArconicRolledProductsCorp_20191217_10-12B_EX-2...,License Agreement


In [21]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

/tmp/ipykernel_5417/3340972191.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.load_local(
    str(FAISS_DIR),
    embeddings,
    allow_dangerous_deserialization=True
)

In [25]:
with open( "/content/drive/MyDrive/legal_ai/vectorstores/bm25.pkl","rb") as f:

    bm25_retriever = pickle.load(f)


In [26]:
print(type(faiss_db))
print(type(bm25_retriever))

<class 'langchain_community.vectorstores.faiss.FAISS'>
<class 'langchain_community.retrievers.bm25.BM25Retriever'>


Reciprocal Hybrid Retrieval

In [27]:
# Reciprocal Rank Fusion
from collections import defaultdict

def reciprocal_rank_fusion(
    results,
    k=60
):

    scores = defaultdict(float)
    doc_map = {}

    for docs in results:

        for rank, doc in enumerate(docs):

            key = (
                doc.metadata["source"],
                doc.page_content[:100]
            )

            scores[key] += 1/(rank+k)

            doc_map[key] = doc

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc_map[key]
        for key, score in ranked
    ]

In [28]:
# Hybrid search
def hybrid_search(
    query,
    k=20
):

    dense_results = faiss_db.similarity_search(
        query,
        k=k
    )

    sparse_results = bm25_retriever.invoke(
        query
    )

    fused_results = reciprocal_rank_fusion(
        [
            dense_results,
            sparse_results
        ]
    )

    return fused_results[:k]

In [30]:
#  verifying the Hybrid Search
query = "force majeure"

results = hybrid_search(
    query,
    k=5
)

print("Results:", len(results))

Results: 5


In [31]:
for i, doc in enumerate(results, start=1):

    print("="*100)

    print(f"RESULT {i}")

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nTYPE:")
    print(
        doc.metadata.get(
            "contract_type",
            "Unknown"
        )
    )

    print("\nTEXT:")
    print(doc.page_content[:600])

RESULT 1

SOURCE:
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt

TYPE:
Maintenance Agreement

TEXT:
17







ARTICLE VII  FORCE MAJEURE

7.1 Procedure.
RESULT 2

SOURCE:
PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement.txt

TYPE:
Hosting Agreement

TEXT:
14.2 The Party prevented from fulfilling its obligations shall on becoming aware of such event inform the other Party in writing of such force majeure event as soon as possible. If the affected Party fails to inform the other Party of the occurrence of a force majeure event as set out in article 14.1 above, then such Party thereafter shall not be entitled to refer such events to force majeure as a reason for non-fulfillment. This obligation does not apply if the force majeure event is known by both Parties or the affected Party is unable to inform the other Party due to the force majeure event
RESULT 3

SOURCE:
VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEMENT.t

### LLM Reranking Theory

Hybrid retrieval improves recall by combining semantic retrieval
(FAISS) and lexical retrieval (BM25).

However, ranking is still based on retrieval scores.

In legal search, clause relevance is often more important than
semantic similarity.

To improve ranking quality, an LLM is used to evaluate
the relationship between:

Question
and
Retrieved Chunk

The LLM assigns a relevance score and the highest scoring
chunks are returned.

In [43]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 1.4 MB/s eta 0:00:00


In [44]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [46]:
import os
os.environ["OPENAI_API_KEY"] ="bedrock-api-key-YmVkcm9jay5hbWF6b25hd3MuY29tLz9BY3Rpb249Q2FsbFdpdGhCZWFyZXJUb2tlbiZYLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFTSUFYNjZWV1o0S1ZXWFY2MklJJTJGMjAyNjA2MjElMkZldS1ub3J0aC0xJTJGYmVkcm9jayUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNjIxVDEyMTY1OVomWC1BbXotRXhwaXJlcz00MzIwMCZYLUFtei1TZWN1cml0eS1Ub2tlbj1JUW9KYjNKcFoybHVYMlZqRUNRYUNtVjFMVzV2Y25Sb0xURWlSekJGQWlCT3JPZFp3RXllbHE4emVOY0pxNmpaajQ2eG1MVGpYUkNUbm5URkg4dlhuUUloQU1OVnc4Qzk0cWU2VndDTiUyRmsyJTJCNlRodFZzekFKYzBzY0k3eVBwS2lSWmVtS3JJRENPMyUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRndFUUFCb01OVFEzTlRFNU5qUTNOVEE1SWd4SENBZ3NuJTJCQlRVeDdaQm5FcWhnUGJCZVolMkI0cFVTY0NSQ0FRblVmR0llWHV2a3hlR0Jmbk9UVzAlMkJLMHV6OFRVJTJCeGdGYSUyQnVta2U3UUUxV3IxTU5GJTJGZTdaZ3JxMllySzlyeFElMkY5R0Fyc2hxTTUlMkJXQjZJcmRzTmVXbG15ZkZzZnVyYWk4ZjhBOSUyQktEUEdrR1o5MEk5cHFrU09veWtzWVZqSWU1d0VKNTJFOVZzUXQ5SVRUdlVlYk9LUHM1NU9DMTczNXZtY2VaWkpWNndOOFlaR0xVeHB1OHpzNnVCd2R2eklUVFNRdHN3eUx1eWRCOXYxcTV3cU51TWxrUW02SlQ1MVhwckJycFZNQTJkZTFYWVFhNzY4RHVWNG5iYU81ZXRaWUFvJTJCbG1MdXJwQzFxa1h2dGdadVBxMm1pZmVKcyUyRmxMeHROcldrJTJCOThVJTJCNnRscFFXWVBIQ3FIOEZYOWJ3JTJGTXk1eUF0N3A3VldsbGFGZmV0VyUyRkZvTjF2eGo4ZG05ZjJKdjRrV0VHR3ZkdFVNRlpscncxQ1IlMkJWN2VxUkxFdWRoMXdsYXQ1RGJ3THRyTExOM2U2aGJtd2UyeDRjdGxHYkdaTVlvTFlXNGZ5UzhjOUNobVFYJTJCTmVrOEJGSDM3MmRxMEpkRFBEaVFBUjdYMG55VnNTa2JNMzZBNUZNVmhtNDMwdk1RZ0N1NlJJTFlsR0xHVDZQdVA2NkhkRHpEWXdzNnpmMFFZNjNnTGpwTFNFWTBCUGQwdndUdGJLdUFvJTJCMzVlT2FmSmZBU1hJMkt6ZElZaWhXJTJGdmVGbmJXZlljdXlIN3VKbkhwNSUyQkptVTdDYVl6OSUyRnZFcU9iWGtEJTJGek4wME5iWFF2ekZTdG84a041cE82dWh6WlBJNm02JTJGSk9aS1RFMHJFMlJFRk9paUs5SnJWNWxUeG9admFlalBqaFZuaERjTnZERlR4TVpQbDFrUmo5QmoyOERDblNsMGZ5NWtib2dlNGxPTmV4c0FzVTRIZklid1VoaDM5ZUJCTDFMeENjcHRFdWVxTzZxQzVjb3NobmVMS0NKMlNhSDJXNCUyRjdhTjRXcVZES2tqaUgzNlB0am5wdHB4cm5iVzFDN0M0TWc2anZ5aFpXQnlJSFQxRmdZTE5YRUJZdlFPbmNTSTQ1Y0txWUdBUjY3QkxrYTNDdDdPZUFKOGVXRSUyQnFKSlpnOFIlMkIyVm9xSmU3Z1N0OUp6TnU4eFBUcGNKSnhrWFpVcTV1VTJGVjNmQ3I3NkZDbU0lMkJWQW9IdGJsc0xabzY1NTNYSmtFcXVLYnl1aFBEcUFkajY2WkJxaFhpdWRqZGIwUk1MTzYlMkZMNGNzOWYzdm1uejJuVXolMkYlMkY0ZWhhbDFWeGclM0QlM0QmWC1BbXotU2lnbmF0dXJlPWNiMTg4MWM0ZWUwYjc1MjQ3MTdjODc3OWNkNzQ1NzUxYmIzOTM4MzVmZWIzMmU1YWI4ZjAwZTRmYmNhZTlhODgmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JlZlcnNpb249MQ=="
os.environ["OPENAI_BASE_URL"]= "https://bedrock-mantle.eu-north-1.api.aws/v1"


In [64]:
llm = ChatOpenAI(
    model="openai.gpt-oss-120b",
    temperature=0
)

In [81]:
ranking_prompt = ChatPromptTemplate.from_template(
"""
You are a legal retrieval expert.

QUESTION:
{question}

RETRIEVED CHUNKS:

{chunks}

Your task:

Rank the 5 most relevant chunks for answering
the QUESTION.

Consider:

1. Direct relevance
2. Legal clause match
3. Completeness
4. Specificity
5. Whether the chunk directly helps answer the question

Return ONLY the chunk numbers in ranked order.

Example:

3,7,1,10,2

Do not explain.
Do not provide reasoning.
Do not write words.
Return only comma-separated chunk numbers.
"""
)

In [82]:
rerank_chain =  ranking_prompt | llm | StrOutputParser()

In [83]:
# Batch LLM Reranking

def batch_llm_rerank(
    question,
    docs,
    top_k=5
):

    formatted_chunks = ""

    for i, doc in enumerate(docs):

        formatted_chunks += f"""

CHUNK {i}

SOURCE:
{doc.metadata.get("source","Unknown")}

TEXT:
{doc.page_content[:800]}

--------------------------------------------------
"""

    response = rerank_chain.invoke(
        {
            "question": question,
            "chunks": formatted_chunks
        }
    )

    print("RAW RANKING RESPONSE:")
    print(response)

    try:

        ranked_indices = [
            int(x.strip())
            for x in response.split(",")
        ]

    except Exception as e:

        print("Parsing Error:", e)

        ranked_indices = list(range(min(top_k, len(docs))))

    reranked_docs = []

    for idx in ranked_indices[:top_k]:

        if idx < len(docs):

            reranked_docs.append(
                docs[idx]
            )

    return reranked_docs

Advanced Retrieval

In [84]:
def advanced_retrieval(
    query,
    retrieve_k=20,
    final_k=5
):

    candidate_docs = hybrid_search(
        query,
        k=retrieve_k
    )

    reranked_docs = batch_llm_rerank(
        query,
        candidate_docs,
        top_k=final_k
    )

    return reranked_docs

In [85]:
query = "force majeure"

results = advanced_retrieval(
    query,
    retrieve_k=20,
    final_k=5
)

print("\nFINAL RESULTS")
print("="*100)

for rank, doc in enumerate(results, start=1):

    print("="*100)

    print("RANK:", rank)

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nTYPE:")
    print(
        doc.metadata.get(
            "contract_type",
            "Unknown"
        )
    )

    print("\nTEXT:")
    print(doc.page_content[:800])

RAW RANKING RESPONSE:
14,2,4,6,13

FINAL RESULTS
RANK: 1

SOURCE:
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt

TYPE:
Maintenance Agreement

TEXT:
"Force Majeure" means any cause or causes not reasonably within the control of the Party claiming suspension and which, by the exercise of  reasonable diligence, such Party is unable to prevent or overcome, including, without limitation, acts of God, acts, omissions to act, and/or delays  in action of federal, state, or local government or any agency thereof, strikes, lockouts, work stoppages, or other industrial disturbances, acts of a  public enemy, sabotage, wars, blockades, insurrections, riots, acts of terror, epidemics, landslides, lightning, earthquakes, fires, storms, storm  warnings, floods, washouts, extreme cold or freezing weather, arrests and restraints of governments and people, civil or criminal disturbances,  interruptions by governmental or court orders, present and future
RANK: 2

SOURCE:
V

In [87]:
query = "confidentiality"

results = advanced_retrieval(
    query,
    retrieve_k=20,
    final_k=5
)

for rank, doc in enumerate(results, start=1):

    print("="*100)

    print("RANK:", rank)

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nTYPE:")
    print(doc.metadata.get("contract_type"))

    print("\nTEXT:")
    print(doc.page_content[:1000])

RAW RANKING RESPONSE:
10,14,16,12,6
RANK: 1

SOURCE:
Sonos, Inc. - Manufacturing Agreement .txt

TYPE:
Manufacturing Agreement

TEXT:
12.0 CONFIDENTIALITY.

12.1. Definition. "Confidential Information" shall mean any information that is transmitted or otherwise provided by or on behalf of the disclosing party, whether orally or in writing, to the receiving party during the course of its performance under this Agreement which is identified as "Confidential" at the time of disclosure or that should reasonably have been understood by the receiving party because of legends or other markings, the circumstances of disclosure or the nature of the information itself, to be proprietary and/or confidential to the disclosing party. All IAC Property, Sonos Property and Future Products, and any information related to such Future Products, shall always be deemed to be Confidential Information of the respective party providing such information. Confidential Information may be disclosed in written or 

Metadata filtering


In [88]:
def filter_by_contract_type(
    docs,
    contract_type
):

    filtered = []

    for doc in docs:

        if (
            doc.metadata.get(
                "contract_type",
                ""
            ).lower()
            ==
            contract_type.lower()
        ):

            filtered.append(doc)

    return filtered

In [89]:
query = "confidentiality"

results = advanced_retrieval(
    query,
    retrieve_k=20,
    final_k=10
)

license_docs = filter_by_contract_type(
    results,
    "License Agreement"
)

print(
    "License Agreement Results:",
    len(license_docs)
)

RAW RANKING RESPONSE:
10,14,16,12,6
License Agreement Results: 1


In [93]:
for doc in license_docs:
    print(
        doc.metadata["source"]
    )
    print("\n")
    print(
        doc.metadata["contract_type"]
    )

    print("\n")
    print(
        doc.page_content[:1000]
    )

IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement.txt


License Agreement


14. Confidentiality.

(a) Confidential Information. "Confidential Information" means all non-public information about the disclosing party's business or activities that is marked or designated by such party as "confidential" or "proprietary" at the time of disclosure or that reasonably would be understood to be confidential given the circumstances of disclosure. Notwithstanding the foregoing, Confidential Information does not include information that: (a) is in or enters the public domain without breach of this Agreement; (b) the receiving party lawfully receives from a third party without restriction on disclosure and without breach of a nondisclosure obligation; (c) the receiving party rightfully knew prior to receiving such information from the disclosing party; or (d) the receiving party develops entirely independently of, and without any access or reference to or use of, any C

Clause Searching

In [94]:
def search_clause(
    clause_name,
    top_k=5
):

    return advanced_retrieval(
        clause_name,
        retrieve_k=20,
        final_k=top_k
    )

In [95]:
results = search_clause(
    "force majeure"
)

for rank, doc in enumerate(results, start=1):

    print("="*100)

    print("RANK:", rank)

    print(
        "\nSOURCE:",
        doc.metadata["source"]
    )

    print(
        "\nTYPE:",
        doc.metadata["contract_type"]
    )

    print(
        "\nTEXT:"
    )

    print(
        doc.page_content[:1000]
    )

RAW RANKING RESPONSE:
14,2,4,13,1
RANK: 1

SOURCE: SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt

TYPE: Maintenance Agreement

TEXT:
"Force Majeure" means any cause or causes not reasonably within the control of the Party claiming suspension and which, by the exercise of  reasonable diligence, such Party is unable to prevent or overcome, including, without limitation, acts of God, acts, omissions to act, and/or delays  in action of federal, state, or local government or any agency thereof, strikes, lockouts, work stoppages, or other industrial disturbances, acts of a  public enemy, sabotage, wars, blockades, insurrections, riots, acts of terror, epidemics, landslides, lightning, earthquakes, fires, storms, storm  warnings, floods, washouts, extreme cold or freezing weather, arrests and restraints of governments and people, civil or criminal disturbances,  interruptions by governmental or court orders, present and future valid orders of any regulatory bo

In [96]:
test_clauses = [

    "force majeure",

    "confidentiality",

    "indemnification",

    "governing law",

    "termination rights",

    "intellectual property"
]

In [100]:
for clause in test_clauses:

    print("\n")
    print("="*120)

    print("CLAUSE:", clause)

    print("="*120)

    results = search_clause(
        clause,
        top_k=3
    )

    for rank, doc in enumerate(results, start=1):

        print(
            f"{rank}. "
            f"{doc.metadata['source']}"
        )



CLAUSE: force majeure
RAW RANKING RESPONSE:
14,2,4,6,13
1. SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt
2. VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEMENT.txt
3. AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt


CLAUSE: confidentiality
RAW RANKING RESPONSE:
16,10,14,12,6
1. LiquidmetalTechnologiesInc_20200205_8-K_EX-10.1_11968198_EX-10.1_Development Agreement.txt
2. Sonos, Inc. - Manufacturing Agreement .txt
3. IdeanomicsInc_20160330_10-K_EX-10.26_9512211_EX-10.26_Content License Agreement.txt


CLAUSE: indemnification
RAW RANKING RESPONSE:
2,6,10,0,7
1. BERKELEYLIGHTS,INC_06_26_2020-EX-10.12-COLLABORATION AGREEMENT.txt
2. NEONSYSTEMSINC_03_01_1999-EX-10.5-DISTRIBUTOR AGREEMENT_New.txt
3. ReynoldsConsumerProductsInc_20191115_S-1_EX-10.18_11896469_EX-10.18_Supply Agreement.txt


CLAUSE: governing law
RAW RANKING RESPONSE:
0,3,4,6,8
1. ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_47